# demo_a2

> Phase 21 (`specs/multiagent/phase21/`, experiment A2): the experience screen (do private records of past contests make guards persist?) and the rule-text screen (does the re-sharing sentence keep liars' first posts uncorrected?), Claude Haiku 5.5. Quick (the fake client and synthetic rows), the pilot, the main run with Gate A2, and the cached rerun, under `A2_PLAN`.

Artifacts (gitignored): `results/multiagent/a2/{a2quick, a2pilot, a2main, a2rerun, p2}/` (`runs.csv`, `p2.csv`, `records.jsonl`, `turns.jsonl`, `timing.jsonl`, `metrics.json`, `report.md`, `a2.png`, `summary.json`). The response cache and the spend ledger are Phase 10's (`results/multiagent/llm/`), under the cap of $250 per calendar month (P21-10).

- **Quick** (normal tests): the pilot team's sequence in all three arms and the pilot's drop-only board, 3 turns, with the fake client (A2's columns, the records, `pilotCheck21`'s plumbing); then the analysis, report and `a2.png` on `syntheticRows21`, in `a2quick/`.
- **Pilot** (`--cell-timeout 0 --flags llm`, `EXPLRL_LLM_LIVE=1`, `EXPLRL_A2_STAGE=pilot`; about $1.90): team 2190's own-record sequence of 6 boards and one drop-only board at 32 agents (seed 990), not analysed. Needs `A2_PLAN` frozen and pushed after the user approved the estimate (P21-10). Then `pilotCheck21` (stop and ask on any failed check), the projection of the main grid (`costRule21`: report and wait) and the power of Gate A2 from the pilot's spread (`power21`).
- **Main** (`--cell-timeout 0 --flags llm`, `EXPLRL_LLM_LIVE=1`, `EXPLRL_A2_STAGE=main`; about $32): the six waves (boards 1-2 once per team) with P2 in the first wave's pool (P21-14). After wave 2: invalid replies under 5% and the manipulation check in 6 of 8 teams, else stop before wave 3 and report P2 with "Gate A2 not read". Needs `A2_PLAN` frozen with the pilot's verdict, the forecasts' SHA-256 and the user's approval of the projection (`A2_PLAN['approved']['main']`).
- **Rerun** (`--flags llm`, `EXPLRL_A2_STAGE=rerun`; $0): every board replayed from the cache into `a2rerun/`, rows compared with the main run's.

In [ ]:
import copy
import hashlib
import json
import os
import re
import shutil
import subprocess
import time

from explRL.a2_grid import A2_PLAN, ARMS21, TEAMS21, costRule21, estimate21, manipulated, p2Grid, pilotCheck21, pilotGrid21
from explRL.a2_report import a2Summaries, runA2Cell
from explRL.a2_seq import _key, runSequences21
from explRL.a2_synth import pilotSigma, power21, syntheticRows21
from explRL.a2_write import figA2, writeA2
from explRL.bench_grid import readCSV, writeCSV
from explRL.coop_grid import coopHash
from explRL.llm_budget import PRICES, Budget
from explRL.llm_grid import HAIKU
from explRL.llm_sched import mergeTiming, timingSummary
from explRL.train import demo_artifact_dir

A2 = 'multiagent/a2'
WORKERS, LLM_WORKERS = min(8, os.cpu_count() or 1), 32
LLM = demo_artifact_dir(None, 'multiagent/llm')
LIVE = os.environ.get('EXPLRL_LLM_LIVE') == '1'
STAGE = os.environ.get('EXPLRL_A2_STAGE', 'pilot')        # what a live run may do: pilot, main or rerun
LIVE_SPEC = dict(kind='live', cache=str(LLM / 'cache'), cap=float(A2_PLAN['cap']), ledger=str(LLM / 'spend.jsonl'))
FORECAST = LLM.parents[2] / A2_PLAN['forecasts']['file']


def gitCommit():
    try:
        return subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'unknown'
    except OSError:
        return 'unknown'


def runA2(mode, teams, arms, spec, p2=(), gate=None, edit=None, workers=None):
    "Sequences (and P2 alongside) through runSequences21 with runA2Cell: runs.csv, p2.csv, records.jsonl, turns.jsonl, timing.jsonl, metrics.json."
    live, out = spec.get('kind') == 'live', demo_artifact_dir(mode, A2)
    t0 = time.time()
    R = runSequences21(teams, arms, spec, out, workers or (LLM_WORKERS if live else WORKERS), edit=edit,
                       alongside=[('p2', list(p2))] if p2 else (), gate=gate, run=runA2Cell)
    wall, logs = time.time() - t0, out / 'turns'
    rows, p2rows = R['rows'], readCSV(out / 'p2.csv') if (out / 'p2.csv').exists() else []
    with open(out / 'turns.jsonl', 'w') as f:
        for p in sorted(logs.glob('*.jsonl')) if logs.exists() else []:
            f.write(p.read_text())
    timing = timingSummary(mergeTiming(logs, out / 'timing.jsonl'), wall)
    new = [r for r in rows if r['arm'] == arms[0] or _key(r)[2] > 2] + p2rows          # each board once (boards 1-2 are copied)
    meta = dict(mode=mode, planHash=coopHash(A2_PLAN), commit=gitCommit(), model=HAIKU, cap=float(spec.get('cap', 0.0)), wall=wall,
                pricesChecked=PRICES[HAIKU]['checked'], cost=float(sum(float(r['cost']) for r in new)), calls=int(sum(float(r['calls']) for r in new)),
                spent=Budget(spec['cap'], spec['ledger']).spent() if spec.get('ledger') else 0.0)
    (out / 'metrics.json').write_text(json.dumps(dict(meta, ran=R['ran'], stopped=R['stopped'], timing=timing), indent=2, sort_keys=True, default=str) + '\n')
    if live:
        for f in ('metrics.json', 'timing.jsonl'):
            if not (out / f.replace('.', '_live.')).exists():
                shutil.copy(out / f, out / f.replace('.', '_live.'))
    print(f"{mode}: {len(rows)} sequence rows, {len(p2rows)} P2 rows, {meta['calls']} calls, ${meta['cost']:.2f} for these runs, "
          f"${meta['spent']:.2f} spent this month, {wall:.0f} s; waves {R['ran']}; stopped: {R['stopped']}")
    return rows, p2rows, meta, R


def analyse(mode, rows, p2rows, meta, plan=A2_PLAN, nboot=None, partial=None):
    "a2Summaries, report.md, a2.png and summary.json in the mode's folder."
    out = demo_artifact_dir(mode, A2)
    S = a2Summaries(rows, p2rows, plan, partial=partial, nboot=nboot)
    writeA2(out / 'report.md', S, meta, plan)
    figA2(out, S)
    (out / 'summary.json').write_text(json.dumps(dict(meta, summary=S), indent=1, sort_keys=True, default=str) + '\n')
    return S


def mainReady():
    """AC-8, P21-8, P21-10 before the main run: A2_PLAN frozen, the pilot's verdict recorded, the forecasts committed with their
    SHA-256 in A2_PLAN, and the user's approval of the pilot's projection recorded (A2_PLAN['approved']['main'])."""
    f = A2_PLAN['forecasts']
    return [m for m, bad in (
        ('A2_PLAN is not frozen', not A2_PLAN['frozen']), ("the pilot's verdict is not recorded", not A2_PLAN.get('pilotVerdict')),
        ("the forecasts are missing or their SHA-256 differs from A2_PLAN's",
         not (f['sha256'] and FORECAST.exists() and hashlib.sha256(FORECAST.read_bytes()).hexdigest() == f['sha256'])),
        ("the user's approval of the pilot's projection is not recorded", not A2_PLAN['approved'].get('main'))) if bad]


def waveTwo(rows, plan=A2_PLAN):
    "The stability check after wave 2: invalid replies under 5% and the manipulation check in 6 of 8 teams (reasons to stop)."
    b12 = [r for r in rows if _key(r)[2] <= 2 and r['arm'] == plan['arms'][0]]
    inv = sum(float(r['invalid']) for r in b12) / (sum(float(r['calls']) for r in b12) or 1)
    man = sum(manipulated([r for r in rows if r['arm'] == 'own'], t) for t in plan['teams'])
    return ([f'invalid replies {inv:.3f} >= 5%'] if inv >= plan['pilotGate']['maxInvalid'] else []) + \
           ([f"manipulation check {man} of {len(plan['teams'])} < {plan['manipulation']['teams']}"] if man < plan['manipulation']['teams'] else [])

In [ ]:
# test_main_ready (AC-8, P21-10): the pilot and main stages refuse until their preconditions hold
import tempfile
from pathlib import Path
assert not A2_PLAN['frozen'] and len(mainReady()) == 4                      # before T5: nothing frozen, verdict, forecasts or approval
_keep = (copy.deepcopy(A2_PLAN), FORECAST)
try:
    FORECAST = Path(tempfile.mkdtemp()) / 'a2_predictions.json'; FORECAST.write_text('{}')
    A2_PLAN.update(frozen=True, pilotVerdict='checks passed', approved=dict(pilot='x', main='y'),
                   forecasts=dict(A2_PLAN['forecasts'], sha256=hashlib.sha256(b'{}').hexdigest()))
    assert mainReady() == []
    for k, v in (('pilotVerdict', None), ('approved', dict(pilot='x', main=None)), ('forecasts', dict(A2_PLAN['forecasts'], sha256='0' * 64))):
        old = A2_PLAN[k]; A2_PLAN[k] = v
        assert len(mainReady()) == 1, k
        A2_PLAN[k] = old
finally:
    A2_PLAN.clear(); A2_PLAN.update(_keep[0]); FORECAST = _keep[1]
assert waveTwo(syntheticRows21()[0]) == [] and len(waveTwo(syntheticRows21(manipTeams=5)[0])) == 1

## Quick: the fake client and synthetic rows

In [ ]:
t0, q = time.time(), demo_artifact_dir('a2quick', A2)
shutil.rmtree(q, ignore_errors=True)
short = lambda c: dict(c, cfg=dict(copy.deepcopy(c['cfg']), turns=3))
P = pilotGrid21()
rows, p2rows, meta, R = runA2('a2quick', P['teams'], list(ARMS21), dict(kind='fake'), p2=[short(c) for c in P['p2']], edit=short)
assert R['ran'] == [(1, 1), (2, 1), (3, 3), (4, 3), (5, 3), (6, 3)] and len(rows) == 18 and len(p2rows) == 1
assert all(k in r for r in rows for k in ('idGuarding', 'idFollowed', 'agents', 'pos', 'seqId', 'volunteers', 'misinformedFirst', 'honestPooled'))
assert 'suppressedHeld' in p2rows[0] and all(int(r['liarOnFailure']) == 0 for r in rows if _key(r)[2] <= 2)
recs = [json.loads(x) for x in (q / 'records.jsonl').read_text().splitlines()]
own = [r for r in rows if r['arm'] == 'own']
C = pilotCheck21(own, [x for x in recs if x['level'].split('|')[0] in ('common', 'own')], p2rows)
assert C['checks']['records'] and C['checks']['boards'] and C['verdict'] == 'ask'      # the record lines render as frozen
print('pilot checks on the fake run (plumbing only):', C['checks'])
rows, p2 = syntheticRows21()
writeCSV(rows, q / 'synthetic.csv')
S = analyse('a2quick', rows, p2, dict(mode='quick', planHash=coopHash(A2_PLAN), commit=gitCommit(), cost=0.0), nboot=200)
assert S['gate']['passes'] and all(S['p2'][N]['supported'] for N in (8, 32))
for f in ('runs.csv', 'p2.csv', 'records.jsonl', 'turns.jsonl', 'metrics.json', 'synthetic.csv', 'report.md', 'a2.png', 'summary.json'):
    assert (q / f).stat().st_size > 0, f
for p in q.iterdir():
    if p.suffix in ('.csv', '.jsonl', '.json', '.md'):
        assert not re.search(r'sk-ant-|x-api-key|authorization|api[_-]?key', p.read_text(), re.I), p
e = estimate21()
print(f"main grid estimate ${e['main']:.2f} (pilot ${e['pilot']:.2f}; total ${e['total']:.2f}, ${e['margin']:.2f} with a 15% margin)")
assert time.time() - t0 < 300
print(open(q / 'report.md').read()[:2000])

## Pilot (`--cell-timeout 0 --flags llm`, live, about $1.90)

Team 2190's own-record sequence and one drop-only board at 32 agents (seed 990), not analysed. `pilotCheck21` (P21-12): invalid replies under 5%; on boards 1-2 both success guards contest and a correct answer follows one of their contests; no lie on a failure guard's clue; the record lines as frozen. Then the projection of the main grid (report and wait) and the power of Gate A2 from the pilot's spread of guarding.

In [ ]:
#| llm
if not LIVE or STAGE != 'pilot':
    print('skipped: needs EXPLRL_LLM_LIVE=1 and EXPLRL_A2_STAGE=pilot')
else:
    assert A2_PLAN['frozen'] and A2_PLAN['approved']['pilot'], 'report the estimate, get the approval, then freeze and push A2_PLAN before the pilot (AC-8, P21-10)'
    P = pilotGrid21()
    print(f"pilot estimate ${estimate21()['pilot']:.2f}")
    rows, p2rows, meta, R = runA2('a2pilot', P['teams'], P['arms'], LIVE_SPEC, p2=P['p2'])
    recs = [json.loads(x) for x in (demo_artifact_dir('a2pilot', A2) / 'records.jsonl').read_text().splitlines()]
    C = pilotCheck21(rows, recs, p2rows)
    cr, pw = costRule21(C['perCall']), power21(pilotSigma(rows))
    demo_artifact_dir('a2pilot', A2).joinpath('report.md').write_text(
        f"# A2 pilot (not analysed)\n\n{json.dumps(dict(checks=C, projection=cr, power=pw, meta=meta), indent=1, default=str)}\n")
    print(f"checks {'pass' if C['ok'] else 'FAIL: stop and ask'} {C['reasons']}; cost per call {C['perCall']}; projected main grid "
          f"${cr['projected']:.2f} against the ${cr['ceiling']:.0f} ceiling: report to the user and wait ({cr['verdict']}); power {pw['power']}")

## Main run and Gate A2 (`--cell-timeout 0 --flags llm`, live, about $32)

Six waves; P2 in the first wave's pool (P21-14). After wave 2 the stability check (`waveTwo`) may stop the run: then P2, the manipulation check and "Gate A2 not read" are reported (`partial='stopped'`).

In [ ]:
#| llm
if not LIVE or STAGE != 'main':
    print('skipped: needs EXPLRL_LLM_LIVE=1 and EXPLRL_A2_STAGE=main')
else:
    bad = mainReady()
    assert not bad, f'the main run waits (AC-8, P21-8, P21-10): {bad}'
    rows, p2rows, meta, R = runA2('a2main', TEAMS21, list(ARMS21), LIVE_SPEC, p2=p2Grid(), gate=waveTwo)
    out = demo_artifact_dir('p2', A2)
    writeCSV(p2rows, out / 'runs.csv')
    gated = R['stopped'] and R['ran'][-1][0] == 2 and bool(waveTwo(rows))        # the stability check stopped it after wave 2
    if R['stopped'] and not gated:
        print(f"stopped ({R['stopped']}): nothing analysed; rerun this stage to resume (finished boards come from the cache)")
    else:
        S = analyse('a2main', rows, p2rows, meta, partial='stopped' if gated else None)
        print(f"stopped: {R['stopped']}; Gate A2: {S['gate']}")
        print(open(demo_artifact_dir('a2main', A2) / 'report.md').read()[:4000])

## Rerun from the cache (`--flags llm`, $0)

Every board replayed through the response cache (the fake client answers only a miss, which then shows as a difference) into `a2rerun/`; its rows must equal the main run's.

In [ ]:
#| llm
if STAGE != 'rerun':
    print('skipped: needs EXPLRL_A2_STAGE=rerun')
else:
    main = demo_artifact_dir('a2main', A2)
    assert (main / 'runs.csv').exists(), 'run the main stage first'
    shutil.rmtree(demo_artifact_dir('a2rerun', A2), ignore_errors=True)
    rows, p2rows, meta, R = runA2('a2rerun', TEAMS21, list(ARMS21), dict(kind='fake', cache=LIVE_SPEC['cache']), p2=p2Grid(), gate=waveTwo)
    rr = demo_artifact_dir('a2rerun', A2)                         # both sides read from CSV (the same number types)
    for a, b in ((readCSV(main / 'runs.csv'), readCSV(rr / 'runs.csv')), (readCSV(main / 'p2.csv'), readCSV(rr / 'p2.csv'))):
        ka = {(_key(r) if 'arm' in r else (r['study'], r['N'], r['seed'])): r for r in a}
        kb = {(_key(r) if 'arm' in r else (r['study'], r['N'], r['seed'])): r for r in b}
        assert ka.keys() == kb.keys(), 'the rerun has other boards'
        diff = [k for k in ka if json.dumps(ka[k], sort_keys=True, default=str) != json.dumps(dict(kb[k]), sort_keys=True, default=str)]
        print(f'{len(ka)} boards, {len(diff)} differ', diff[:3])